In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Affinity reported from Win32 API: 0, 1, 2, 3, 4, 5, 6, 7;
Affinity reported from Win32 API: 0, 1, 2, 3, 4, 5, 6, 7;
Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Databases loaded: 
Capacity: 0
Count: 0



Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 217
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 104
   at Submission#35.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble2D")
// OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble3D")

Opening existing database 'P:\hpccluster\RisingBubble2D'.


{ Session Count = 1; Grid Count = 2; Path = P:\hpccluster\RisingBubble2D }

In [ ]:
var sessions = databases.Pick(0).Sessions;
sessions

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	08/27/2024 14:57:24	d9c4a861...


In [ ]:
var sess = sessions.Pick(0);
sess

RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	08/27/2024 14:57:24	d9c4a861...

In [ ]:
var tStep_fail = sess.Timesteps.Last();
tStep_fail

 { Time-step: 718; Physical time: 1.434000000000001s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, GravityY#A, GravityY#B, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }

In [ ]:
var tStep_OK = sess.Timesteps.Pick(sess.Timesteps.Count()-2);
tStep_OK

 { Time-step: 717; Physical time: 1.434000000000001s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, GravityY#A, GravityY#B, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }

## Evaluate Interface Continuity

In [ ]:
using BoSSS.Foundation.Quadrature;

In [ ]:
var tStep = tStep_fail;
// var tStep = tStep_OK;

In [ ]:
SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
GridData grdData = (GridData)phi.GridDat; 
LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
levelSet.Acc(1.0, phi); 
LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
LsTrk.UpdateTracker(tStep.PhysicalTime);

In [ ]:
//LsTrk.Regions.GetCutCellMask().ItemEnum.Count()

In [ ]:
int order = phi.Basis.Degree;
XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;
EdgeQuadratureScheme SurfaceElement_Edge = SchemeHelper.Get_SurfaceElement_EdgeQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
//SurfaceElement_Edge.Domain.ItemEnum.Count()

In [ ]:
EdgeMask CutCellBoundaryEdgeMask = LsTrk.Regions.GetCutCellMask().AllEdges().Except(LsTrk.Regions.GetCutCellMask().GetAllInnerEdgesMask());
//CutCellBoundaryEdgeMask.ItemEnum.Count()

In [ ]:
var factory = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadFactoryHelper.GetSurfaceElement_BoundaryRuleFactory(0, LsTrk.GridDat.Grid.RefElements[0]);
EdgeQuadratureScheme SurfaceElement_BoundaryEdge = new EdgeQuadratureScheme(factory, CutCellBoundaryEdgeMask);

In [ ]:
double result = 0;
int D = LsTrk.GridDat.SpatialDimension;
EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SurfaceElement_BoundaryEdge.Compile(LsTrk.GridDat, 0),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

        // inner quadrature node
        NodeSet Enode_l = QR.Nodes;
        int trf = LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0, 0];
        NodeSet Vnode_l = Enode_l.GetVolumeNodeSet(LsTrk.GridDat, trf, false);
        NodeSet Vnode_g = Vnode_l.CloneAs();
        int cell = LsTrk.GridDat.Edges.CellIndices[i0, 0];
        LsTrk.GridDat.TransformLocal2Global(Vnode_l, Vnode_g, cell);
        if (D == 2)
            Console.WriteLine("inner quadrature node: ({0},{1})", Vnode_g[0, 0], Vnode_g[0, 1]);
        else 
            Console.WriteLine("inner quadrature node: ({0},{1},{2})", Vnode_g[0, 0], Vnode_g[0, 1], Vnode_g[0, 2]);

        for(int i = 0; i < length; i++) {
            EvalResult[i, 0, 0] = 1;    
        }

    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            result += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
result

inner quadrature node: (0.34472956972506547,0.6000000000000001)
inner quadrature node: (0.3500000000000001,0.6004963167011994)
inner quadrature node: (0.6499999999999999,0.6004963024732727)
inner quadrature node: (0.6552702631213384,0.6000000000000001)


4

In [ ]:
CellQuadratureScheme ContactLineVolumeScheme = SchemeHelper.GetContactLineQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
ContactLineVolumeScheme.Domain.ItemEnum.Count()

0

## Continuity projection

In [ ]:
using BoSSS.Solution.LevelSetTools;

In [ ]:
SinglePhaseField phiDG = (SinglePhaseField)tStep.GetField("PhiDG");

In [ ]:
ContinuityProjection preEnforcer = new ContinuityProjection(
    phi.Basis,
    phiDG.Basis,
    LsTrk.GridDat,
    ContinuityProjectionOption.ConstrainedDG);